# Decade and face-count stratified page sampling

This notebook creates a 600-image manifest for face annotation from the cleaned Economist face-detection data. The sampling unit is a unique source scan (`source_scan_id`), rather than an individual detected-face record or a literal generated detector filename.

Sampling is performed in two stages: first across decades in proportion to each decade's unique-page population, with the smallest adjustment needed to make the bucket minimums feasible, then across five face-count buckets within each decade in proportion to the bucket population. Each nonempty bucket receives at least four sampled pages where its population permits it. The minimum acts as a lower bound inside the allocation rather than as a preliminary sample, avoiding additional inflation of small buckets. Integer allocations use the largest-remainder (Hamilton) method and a fixed seed makes the result reproducible.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import jsonschema
import numpy as np
import pandas as pd

INPUT_CSV = Path("../../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned.csv")
JOINED_IMAGE_MANIFEST = Path("../../../data/processed/full_pages_1940_2007_joined_manifest.csv")
MANIFEST_SCHEMA = Path("annotation_app_manifest_schema_v1.json")
OUTPUT_MANIFEST = Path("../../../data/annotations/economist_decade_face_count_stratified_600_v3_manifest.json")
TASK_ID = "economist_decade_face_count_stratified_600_v3"

SAMPLE_SIZE = 600
BLOCK_SIZE = 30
MINIMUM_PER_NONEMPTY_BUCKET = 4
RANDOM_SEED = 20260802
EXPECTED_FACE_RECORDS = 63_832
EXPECTED_LITERAL_DETECTOR_FILENAMES = 34_958
EXPECTED_UNIQUE_PAGES = 33_047

FACE_BUCKETS = ["1", "2", "3-5", "6-11", "12+"]
FACE_BUCKET_BINS = [0, 1, 2, 5, 11, np.inf]

assert SAMPLE_SIZE > 0
assert BLOCK_SIZE > 0
assert MINIMUM_PER_NONEMPTY_BUCKET > 0
assert INPUT_CSV.is_file(), f"Missing input CSV: {INPUT_CSV}"
assert JOINED_IMAGE_MANIFEST.is_file(), f"Missing joined-image manifest: {JOINED_IMAGE_MANIFEST}"
assert MANIFEST_SCHEMA.is_file(), f"Missing manifest schema: {MANIFEST_SCHEMA}"


## Load and validate the face records

The source file contains one row per retained face detection. Generated detector suffixes are removed to recover `source_scan_id`, the page-scoped identity established by the deduplication workflow. A single scan uses its canonical page filename; a two-page scan replaces the comma-separated page numbers with the underscore-separated filename of the joined JPEG. Every canonical name is checked against the joined-image manifest. Archive pages with no retained face detection are outside the population, and face counts are detector records rather than manually verified counts.

In [ ]:
faces = pd.read_csv(INPUT_CSV, dtype={"Filename": "string"})
joined_images = pd.read_csv(
    JOINED_IMAGE_MANIFEST, dtype={"output_filename": "string", "scan_id": "string"}
)

assert "Filename" in faces.columns
assert faces["Filename"].notna().all(), "Every face record must identify a detector filename."
assert {"output_filename", "scan_id", "output_type"}.issubset(joined_images.columns)

faces["source_scan_id"] = (
    faces["Filename"].str.replace(r"\.[^.]+$", "", regex=True).str.split("_", n=1).str[0]
)
assert faces["source_scan_id"].str.fullmatch(
    r"\d{4}-\d{4}-\d{4}(?:,\d{4})?"
).all(), "Every detector filename must yield a one- or two-page source_scan_id."

pages = faces.groupby("source_scan_id", as_index=False).size().rename(columns={"size": "face_count"})
pages["Filename"] = pages["source_scan_id"].str.replace(",", "_", regex=False) + ".jpg"
pages["page_type"] = np.where(pages["source_scan_id"].str.contains(","), "double", "single")
year = pages["source_scan_id"].str.extract(r"^(\d{4})-", expand=False)
assert year.notna().all(), "Every source_scan_id must begin with a four-digit year and a hyphen."

pages["year"] = year.astype(int)
pages["decade"] = (pages["year"] // 10) * 10
pages["face_bucket"] = pd.cut(
    pages["face_count"],
    bins=FACE_BUCKET_BINS,
    labels=FACE_BUCKETS,
    include_lowest=True,
    ordered=True,
)

assert joined_images["scan_id"].is_unique
assert joined_images["output_filename"].is_unique
canonical_check = pages[["source_scan_id", "Filename", "page_type"]].merge(
    joined_images[["scan_id", "output_filename", "output_type"]],
    left_on="source_scan_id",
    right_on="scan_id",
    how="left",
    validate="one_to_one",
)
assert canonical_check["output_filename"].notna().all()
assert canonical_check["Filename"].eq(canonical_check["output_filename"]).all()
assert canonical_check["page_type"].eq(
    canonical_check["output_type"].map({"single": "single", "joined_two_page": "double"})
).all()
assert set(pages["Filename"]) == set(joined_images["output_filename"])

assert len(faces) == EXPECTED_FACE_RECORDS, f"Expected {EXPECTED_FACE_RECORDS:,} face records, found {len(faces):,}."
assert faces["Filename"].nunique() == EXPECTED_LITERAL_DETECTOR_FILENAMES
assert len(pages) == EXPECTED_UNIQUE_PAGES, f"Expected {EXPECTED_UNIQUE_PAGES:,} unique source scans, found {len(pages):,}."
assert pages["source_scan_id"].is_unique
assert pages["Filename"].is_unique
assert pages["face_count"].ge(1).all()
assert pages["face_bucket"].notna().all()
assert SAMPLE_SIZE <= len(pages), "The requested sample exceeds the unique-source-scan population."

print(f"Face records: {len(faces):,}")
print(f"Literal detector filenames: {faces['Filename'].nunique():,}")
print(f"Unique source scans / canonical images: {len(pages):,}")
print(f"Years represented: {pages['year'].min()}–{pages['year'].max()}")


## Allocate the sample

Hamilton allocation first distributes the 600 pages across decades according to the number of unique pages. The sum of the bucket minimums acts as a decade-level lower bound, which minimally raises a decade allocation if its unconstrained quota cannot support those minimums. Within each decade, `min(4, bucket population)` is imposed as a lower bound. Buckets whose recalculated proportional quota falls below that bound are fixed at the minimum; the remaining places are then reapportioned among the other buckets in proportion to their original population sizes. This iterative bounded-Hamilton procedure guarantees rare-bucket coverage without giving those buckets both a preliminary minimum and an additional share of the residual sample.

In [ ]:
def hamilton_allocate(capacities: pd.Series, total: int) -> pd.Series:
    """Allocate integer places proportionally, with stable largest-remainder tie-breaking."""
    capacities = capacities.astype(int)
    assert capacities.ge(0).all()
    assert 0 <= total <= capacities.sum()

    if total == 0:
        return pd.Series(0, index=capacities.index, dtype=int)

    quotas = capacities / capacities.sum() * total
    allocation = np.floor(quotas).astype(int)
    remaining_places = total - int(allocation.sum())

    remainders = (quotas - allocation).sort_values(ascending=False, kind="stable")
    allocation.loc[remainders.index[:remaining_places]] += 1

    assert int(allocation.sum()) == total
    assert allocation.le(capacities).all()
    return allocation


def lower_bounded_hamilton_allocate(
    populations: pd.Series, total: int, lower_bounds: pd.Series
) -> pd.Series:
    """Allocate proportionally while treating minimums as lower bounds, not additions."""
    populations = populations.astype(int)
    lower_bounds = lower_bounds.astype(int).reindex(populations.index)
    assert populations.ge(0).all()
    assert lower_bounds.ge(0).all()
    assert lower_bounds.le(populations).all()
    assert lower_bounds.sum() <= total <= populations.sum()

    allocation = pd.Series(0, index=populations.index, dtype=int)
    active = populations.gt(0)
    remaining_total = total

    while active.any():
        active_populations = populations.loc[active]
        active_lower_bounds = lower_bounds.loc[active]
        assert active_lower_bounds.sum() <= remaining_total <= active_populations.sum()

        recalculated_quotas = active_populations / active_populations.sum() * remaining_total
        forced_to_minimum = recalculated_quotas.lt(active_lower_bounds)

        if not forced_to_minimum.any():
            allocation.loc[active] = hamilton_allocate(active_populations, remaining_total)
            remaining_total = 0
            break

        forced_index = forced_to_minimum.index[forced_to_minimum]
        allocation.loc[forced_index] = lower_bounds.loc[forced_index]
        remaining_total -= int(lower_bounds.loc[forced_index].sum())
        active.loc[forced_index] = False

    assert remaining_total == 0
    assert int(allocation.sum()) == total
    assert allocation.ge(lower_bounds).all()
    assert allocation.le(populations).all()
    return allocation


decade_population = pages.groupby("decade", sort=True).size().rename("population_images")

strata_index = pd.MultiIndex.from_product(
    [decade_population.index, FACE_BUCKETS], names=["decade", "face_bucket"]
)
strata = (
    pages.groupby(["decade", "face_bucket"], observed=False)
    .size()
    .reindex(strata_index, fill_value=0)
    .rename("population_images")
    .reset_index()
)
strata["minimum_sample"] = strata["population_images"].clip(upper=MINIMUM_PER_NONEMPTY_BUCKET)
minimum_by_decade = strata.groupby("decade")["minimum_sample"].sum()

decade_samples = lower_bounded_hamilton_allocate(
    populations=decade_population,
    total=SAMPLE_SIZE,
    lower_bounds=minimum_by_decade,
).rename("sample_images")
assert (decade_samples >= minimum_by_decade).all()

strata["sample_images"] = 0
for decade, decade_total in decade_samples.items():
    decade_mask = strata["decade"].eq(decade)
    decade_allocation = lower_bounded_hamilton_allocate(
        populations=strata.loc[decade_mask, "population_images"],
        total=int(decade_total),
        lower_bounds=strata.loc[decade_mask, "minimum_sample"],
    )
    strata.loc[decade_mask, "sample_images"] = decade_allocation.to_numpy()

assert int(strata["sample_images"].sum()) == SAMPLE_SIZE
assert strata["sample_images"].le(strata["population_images"]).all()
assert (strata.groupby("decade")["sample_images"].sum() == decade_samples).all()

allocation_report = strata.merge(decade_population.rename("decade_population"), on="decade")
allocation_report = allocation_report.merge(decade_samples.rename("decade_sample"), on="decade")
allocation_report["population_share_within_decade"] = (
    allocation_report["population_images"] / allocation_report["decade_population"]
)
allocation_report["sample_share_within_decade"] = (
    allocation_report["sample_images"] / allocation_report["decade_sample"]
)
allocation_report["relative_to_decade"] = (
    allocation_report["sample_share_within_decade"]
    / allocation_report["population_share_within_decade"].where(allocation_report["population_images"].gt(0))
)

display(
    allocation_report[
        ["decade", "face_bucket", "population_images", "minimum_sample", "sample_images"]
    ]
)


## Draw unique pages and build the manifest

Random values are generated once for each unique source scan and used to select the allocated number of images per stratum. The manifest is then ordered with a deficit-based proportional interleaving heuristic: at every position, it chooses the available stratum that is furthest below its share in the planned 600-image sample. This aims to keep prefixes close to the final sample's decade × face-count distribution, but it does not guarantee globally optimal or monotonically decreasing distance at every block boundary. Any deliberate divergence caused by the minimum-bucket rule is reported below. Page type is derived from `source_scan_id`, while the manifest uses canonical single-page or joined-double-page JPEG filenames. No local image paths are included.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
pages["selection_random"] = rng.random(len(pages))

selection_frame = pages.merge(
    strata[["decade", "face_bucket", "sample_images"]],
    on=["decade", "face_bucket"],
    how="left",
    validate="many_to_one",
)
selection_frame = selection_frame.sort_values(
    ["decade", "face_bucket", "selection_random"], kind="stable"
).copy()
selection_frame["within_stratum_rank"] = (
    selection_frame.groupby(["decade", "face_bucket"], observed=True).cumcount() + 1
)
sampled_pages = selection_frame.loc[
    selection_frame["within_stratum_rank"] <= selection_frame["sample_images"]
].copy()

assert len(sampled_pages) == SAMPLE_SIZE
assert sampled_pages["Filename"].is_unique
expected_stratum_samples = strata.set_index(["decade", "face_bucket"])["sample_images"]
actual_stratum_samples = sampled_pages.groupby(["decade", "face_bucket"], observed=True).size()
assert actual_stratum_samples.reindex(expected_stratum_samples.index, fill_value=0).equals(expected_stratum_samples)

# Order the selected pages so every prefix follows the planned 600-page stratum shares.
pool = sampled_pages.sort_values(
    ["decade", "face_bucket", "selection_random"], kind="stable"
).copy()
pool["within_stratum_rank"] = (
    pool.groupby(["decade", "face_bucket"], observed=True).cumcount() + 1
)

stratum_population = strata.set_index(["decade", "face_bucket"])["population_images"]
stratum_sample = strata.set_index(["decade", "face_bucket"])["sample_images"]
stratum_share = stratum_sample / stratum_sample.sum()
remaining = stratum_sample.copy()
selected_so_far = pd.Series(0, index=stratum_sample.index, dtype=int)
schedule = []

for position in range(1, SAMPLE_SIZE + 1):
    deficit = position * stratum_share - selected_so_far
    available = deficit.loc[remaining.gt(0)].sort_values(ascending=False, kind="stable")
    chosen_stratum = available.index[0]
    schedule.append(chosen_stratum)
    selected_so_far.loc[chosen_stratum] += 1
    remaining.loc[chosen_stratum] -= 1

assert remaining.eq(0).all()
schedule = pd.DataFrame(schedule, columns=["decade", "face_bucket"])
schedule["within_stratum_rank"] = (
    schedule.groupby(["decade", "face_bucket"], observed=True).cumcount() + 1
)
schedule["manifest_position"] = np.arange(1, SAMPLE_SIZE + 1)

sampled_pages = schedule.merge(
    pool,
    on=["decade", "face_bucket", "within_stratum_rank"],
    how="left",
    validate="one_to_one",
).sort_values("manifest_position", kind="stable").reset_index(drop=True)

assert sampled_pages["Filename"].notna().all()
assert sampled_pages["Filename"].is_unique

manifest = {
    "task_id": TASK_ID,
    "block_size": BLOCK_SIZE,
    "metadata": {
        "input_csv": INPUT_CSV.name,
        "input_face_records": int(len(faces)),
        "input_literal_detector_filenames": int(faces["Filename"].nunique()),
        "input_unique_pages": int(len(pages)),
        "joined_image_manifest": JOINED_IMAGE_MANIFEST.name,
        "sampling_unit": "unique source_scan_id represented in the cleaned face-detection CSV",
        "population_scope": (
            "pages with at least one retained face detection; detected face counts are not "
            "manually verified face counts"
        ),
        "population_size": int(len(pages)),
        "sample_size": SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "decade_allocation": (
            "lower-bounded Hamilton allocation weighted by unique-page population; "
            "decade lower bounds equal the sum of feasible bucket minimums"
        ),
        "within_decade_allocation": (
            "iterative lower-bounded Hamilton allocation weighted by face-count-bucket "
            f"population, with a minimum of up to {MINIMUM_PER_NONEMPTY_BUCKET} pages "
            "per nonempty bucket"
        ),
        "face_count_buckets": FACE_BUCKETS,
        "page_type_rule": (
            "source_scan_id values containing a comma are double pages; their canonical "
            "joined JPEG filenames replace the comma with an underscore"
        ),
        "manifest_ordering": (
            "deficit-based proportional interleaving heuristic against planned 600-page "
            "decade × face-count-bucket shares"
        ),
    },
    "images": [
        {
            "image_id": Path(row.Filename).stem,
            "filename": row.Filename,
            "page_type": row.page_type,
            "metadata": {
                "source_scan_id": row.source_scan_id,
                "year": int(row.year),
                "decade": int(row.decade),
                "face_count": int(row.face_count),
                "face_count_bucket": str(row.face_bucket),
                "manifest_position": int(row.manifest_position),
            },
        }
        for row in sampled_pages.itertuples(index=False)
    ],
}

OUTPUT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_MANIFEST.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
    handle.write("\n")

print(f"Wrote {len(manifest['images']):,} pages in {len(manifest['images']) // BLOCK_SIZE} complete blocks: {OUTPUT_MANIFEST}")


## Verify the manifest and report oversampling

`relative_to_decade` compares each stratum's sampled share with its population share within that decade. Values above one identify overrepresentation caused by the lower-bound constraint and integer apportionment. The overall report compares each bucket's share of all sampled pages with its share of the full unique-page population. The prefix-balance report compares each 30-page boundary with the planned 600-page distribution: zero would be an exact distributional match, so smaller values indicate closer alignment.

In [ ]:
with MANIFEST_SCHEMA.open(encoding="utf-8") as handle:
    schema = json.load(handle)
with OUTPUT_MANIFEST.open(encoding="utf-8") as handle:
    written_manifest = json.load(handle)

jsonschema.validate(instance=written_manifest, schema=schema)
print("Full JSON Schema validation passed.")

assert set(schema["required"]).issubset(written_manifest)
assert written_manifest["block_size"] == BLOCK_SIZE
assert len(written_manifest["images"]) == SAMPLE_SIZE
assert len(written_manifest["images"]) % BLOCK_SIZE == 0
assert all(set(schema["properties"]["images"]["items"]["required"]).issubset(image) for image in written_manifest["images"])
assert all(re.fullmatch(r"[^/\\]+\.jpe?g", image["filename"]) for image in written_manifest["images"])
assert all("path" not in image for image in written_manifest["images"])
assert {image["page_type"] for image in written_manifest["images"]}.issubset({"single", "double"})
assert len({image["filename"] for image in written_manifest["images"]}) == SAMPLE_SIZE
assert {image["filename"] for image in written_manifest["images"]}.issubset(
    set(joined_images["output_filename"])
)

joint_target_share = stratum_sample / SAMPLE_SIZE
decade_target_share = decade_samples / SAMPLE_SIZE
bucket_target_share = (
    strata.groupby("face_bucket", observed=False)["sample_images"].sum().reindex(FACE_BUCKETS) / SAMPLE_SIZE
)

prefix_rows = []
for prefix_size in range(BLOCK_SIZE, SAMPLE_SIZE + 1, BLOCK_SIZE):
    prefix = sampled_pages.iloc[:prefix_size]
    joint_sample_share = (
        prefix.groupby(["decade", "face_bucket"], observed=True).size()
        .reindex(joint_target_share.index, fill_value=0)
        / prefix_size
    )
    decade_sample_share = prefix.groupby("decade").size().reindex(decade_target_share.index, fill_value=0) / prefix_size
    bucket_sample_share = (
        prefix.groupby("face_bucket", observed=False).size().reindex(FACE_BUCKETS, fill_value=0) / prefix_size
    )
    prefix_rows.append(
        {
            "annotated_pages_if_stopped": prefix_size,
            "joint_stratum_tv_distance": 0.5 * (joint_sample_share - joint_target_share).abs().sum(),
            "decade_tv_distance": 0.5 * (decade_sample_share - decade_target_share).abs().sum(),
            "face_bucket_tv_distance": 0.5 * (bucket_sample_share - bucket_target_share).abs().sum(),
        }
    )
prefix_balance_report = pd.DataFrame(prefix_rows).round(4)

allocation_report["oversampled_relative_to_decade"] = allocation_report["relative_to_decade"].gt(1)
within_decade_ratio = (
    allocation_report.pivot(index="decade", columns="face_bucket", values="relative_to_decade")
    .reindex(columns=FACE_BUCKETS)
)
within_decade_display = within_decade_ratio.map(
    lambda value: f"{value:.2f}×" if value > 1 else "—"
)
within_decade_display.index.name = "Decade"
within_decade_display.columns.name = "Face-count bucket"

overall_bucket_report = (
    allocation_report.groupby("face_bucket", observed=False)[["population_images", "sample_images"]]
    .sum()
    .reindex(FACE_BUCKETS)
    .reset_index()
)
overall_bucket_report["population_share"] = overall_bucket_report["population_images"] / len(pages)
overall_bucket_report["sample_share"] = overall_bucket_report["sample_images"] / SAMPLE_SIZE
overall_bucket_report["relative_to_whole_population"] = (
    overall_bucket_report["sample_share"] / overall_bucket_report["population_share"]
)
overall_bucket_report["oversampled_relative_to_whole_population"] = (
    overall_bucket_report["relative_to_whole_population"].gt(1)
)

print("Balance against the planned 600-page distribution at every stopping point (lower is better):")
display(
    prefix_balance_report.rename(
        columns={
            "annotated_pages_if_stopped": "Pages annotated",
            "joint_stratum_tv_distance": "Joint distance",
            "decade_tv_distance": "Decade distance",
            "face_bucket_tv_distance": "Bucket distance",
        }
    )
)

print("Within-decade oversampling (sample share ÷ population share; — means not oversampled):")
display(within_decade_display)

print("Overall face-count bucket comparison:")
overall_bucket_display = pd.DataFrame(
    {
        "Face-count bucket": overall_bucket_report["face_bucket"],
        "Population": overall_bucket_report.apply(
            lambda row: f"{int(row.population_images):,} ({row.population_share:.2%})", axis=1
        ),
        "Sample": overall_bucket_report.apply(
            lambda row: f"{int(row.sample_images):,} ({row.sample_share:.2%})", axis=1
        ),
        "Sampling ratio": overall_bucket_report["relative_to_whole_population"].map(
            lambda value: f"{value:.2f}×"
        ),
        "Result": overall_bucket_report["oversampled_relative_to_whole_population"].map(
            lambda value: "Oversampled" if value else "Not oversampled"
        ),
    }
)
display(overall_bucket_display)


## Conclusion

The manifest contains 600 distinct page images drawn from pages represented in the cleaned face-detection CSV and arranged as 20 blocks of 30. The final verification cell identifies any deliberate overrepresentation caused by the minimum-per-bucket rule, both within decades and across the whole detector-derived population.